<a href="https://colab.research.google.com/github/vermilion000/AIAgents-nd-LLMs/blob/main/DeepEval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# **1.Import libraries**:
Import required **Testcase** templetes,**Metrics** and **Benchmarks**.


```
from deepeval import assert_test
from deepeval.test_case import (
  LLMTestCase,
  MCPServer,
  ConversationalTestCase, Turn,
  LLMTestCaseParams)
from deepeval.tracing import observe   
from deepeval.metrics import (
    AnswerRelevancyMetric ,PromptAlignmentMetric,SummarizationMetric,
    HallucinationMetric,FaithfulnessMetric,
    MCPUseMetric,
    TurnRelevancyMetric,KnowledgeRetentionMetric
    GEval)
from deepeval.benchmarks import MMLU
from deepeval.benchmarks.tasks import MMLUTask
```



# **2.Defining Metrics**

Here are the most commonly used DeepEval metrics with one-line explanations:


### **RAG Metrics**

* **Answer Relevancy** – Checks whether the generated answer actually addresses the user's question. ([DeepEval][1])
* **Faithfulness** – Checks whether the answer is supported by the retrieved context and avoids hallucinations. ([DeepEval][1])
* **Contextual Relevancy** – Measures how relevant the retrieved documents are to the query. ([DeepEval][2])
* **Contextual Precision** – Measures whether the retrieved context contains mostly useful information with minimal noise. ([DeepEval][1])
* **Contextual Recall** – Measures whether the retrieved context contains all information needed to answer the query. ([DeepEval][1])


### **Agent Metrics**

* **Task Completion** – Evaluates whether the agent successfully completed the requested task. ([DeepEval][1])
* **Tool Correctness** – Checks whether the agent selected and used the correct tools. ([DeepEval][1])
* **Argument Correctness** – Verifies that tool arguments were accurate and appropriate. ([DeepEval][1])
* **Step Efficiency** – Measures whether the agent reached the solution with minimal unnecessary steps. ([DeepEval][1])
* **Plan Adherence** – Checks whether execution followed the intended plan. ([DeepEval][1])
* **Plan Quality** – Evaluates how good the generated plan is for solving the task. ([DeepEval][1])


### **Safety Metrics**

* **Hallucination** – Detects unsupported or fabricated claims.
* **Toxicity** – Detects harmful, offensive, or abusive content.
* **Bias** – Detects unfair or prejudiced outputs.
* **PII Leakage** – Detects exposure of sensitive personal information.


### **General LLM Metrics**

* **GEval** – Scores output against custom evaluation criteria written in natural language. ([DeepEval][1])
* **Prompt Alignment** – Checks whether the response follows the prompt instructions.
* **Summarization** – Evaluates summary quality, coverage, and correctness.
* **Conversation Completeness** – Measures whether a chatbot fully addressed the conversation.


### **Quick mapping for RAG debugging**

| Problem                              | Metric               |
| ------------------------------------ | -------------------- |
| Wrong answer to question             | Answer Relevancy     |
| Hallucinating facts                  | Faithfulness         |
| Retriever fetching irrelevant chunks | Contextual Relevancy |
| Too much retrieval noise             | Contextual Precision |
| Missing important documents          | Contextual Recall    |



**Note**:- Enable/Uncomment which ever metric is relevent
Arguments are just threshold,model to act aas LLm as judge.


Use @observe for llm tracing .It will trace and save data in same directory .We can use Confidential AI to analyse.

In [ ]:
relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini")  # LLMs response relevancy with Retrieved Text

#  PromptAlignmentMetric = PromptAlignmentMetric(                      #LLMs response relevancy with user Prompt
#     prompt_instructions=["Reply in all uppercase"],
#     model="gpt-4o-mini",
#     include_reason=True
# )

# metric = SummarizationMetric(                            #LLM Summarization skill.Alignment_score whether avoids hallucinations,Covergae_score whether all importnant data is generated.
#     threshold=0.7,
#     model="gpt-4o-mini",
# )

# metric = HallucinationMetric(threshold=0.5)

# metric = FaithfulnessMetric(                    # how truthfull the geneeretaed output is to retrieved text
#     threshold=0.7,
#     model="gpt-4o-mini",
#     include_reason=True
# )

# metric = TurnRelevancyMetric()
# metric = KnowledgeRetentionMetric()


# **3.Writting TestCases** :

*   Test Case can have Query,Output,Retreived _conetxt.
*   Bechmarks comes with their own testcases.See next section **"4.Benchmark Testing"** .
*   Also take attention to conversational testcase for Chatbot Evaluation using TurnReleveancy,KnowledgeRetenntion.They have turn based structure.



```
test_case = LLMTestCase(
        input="Can I return these shoes after 30 days?",
        actual_output="Yes, you can return them. We offer a 30-day full refund. Do you have your original receipt?",
        retrieval_context=[
            "All customers are eligible for a 30-day full refund at no extra cost.",
            "Returns are only accepted within 30 days of purchase.",
        ],
    )

# test_case = ConversationalTestCase(
#     turns=[
#         Turn(role="user", content="Hello, how are you?"),
#         Turn(role="assistant", content="I'm doing well, thank you!"),
#         Turn(role="user", content="How can I help you today?"),
#         Turn(role="assistant", content="I'd like to buy a ticket to a Coldplay concert."),
#     ]
# )

metric.measure(test_case)                                   #evalutes only one metric ,one testcase .The "metric.score,metric.reason" stores score and reason
print("metric.score:", metric.score)
print("metric.reason:", metric.reason)

# or evaluate test cases in bulk
evaluate([test_case], [metric])
```




# **4.Benchmark Testing** :

 **MMLU** **(** **Massive Multitask Language Understanding** **)** is a benchmark commonly used to evaluate large language models through multiple-choice questions. Covering 57 subjects ranging from math and history to law and ethics, it provides a thorough assessment of an LLM’s knowledge and reasoning skills. Its wide subject coverage and carefully designed questions have made MMLU a gold standard for measuring model performance.

 *Evaluation can be done with metric object.




```
##MMLU benchmark adding benchmark and testcase.
benchmark = MMLU(
        tasks=[MMLUTask.HIGH_SCHOOL_COMPUTER_SCIENCE],
        n_shots=1
    )
    benchmark.evaluate(model=custom_model, batch_size=1)
```



## **5.MCP Evaluation using DeepEval** :
LLMtestcase should contain server information for metric to track mcp call and uses.


```
from fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("Simple Echo Server")


test_case = LLMTestCase(
    input="Hello", # Your input here
    actual_output="Hello", # Your LLM app's final output here
    mcp_servers=[MCPServer(server_name="Simple Echo Server")] # Your MCP server's data
    # MCP primitives used (if any)
)

metric = MCPUseMetric()
```



# **6.G - Eval Metric:**

G-Eval (Generative Evaluation) is DeepEval's flexible LLM-as-a-judge metric that lets you evaluate outputs using custom criteria written in natural language instead of hard-coded rules.


**G-Eval is used exactly like other DeepEval metrics.** The only difference is that you define the evaluation criteria yourself.

The values specified in `evaluation_params` are automatically pulled from the `LLMTestCase` you pass to `measure()` or `evaluate()`.

What happens internally:

* `INPUT` → `"What is the capital of France?"`
* `ACTUAL_OUTPUT` → `"Paris is the capital of France."`
* DeepEval builds an evaluation prompt using those fields and asks the judge LLM to score it.

---

### How `evaluation_params` maps to `LLMTestCase`

| evaluation_params | Must exist in LLMTestCase |
| ----------------- | ------------------------- |
| INPUT             | `input=`                  |
| ACTUAL_OUTPUT     | `actual_output=`          |
| EXPECTED_OUTPUT   | `expected_output=`        |
| RETRIEVAL_CONTEXT | `retrieval_context=`      |
| CONTEXT           | `context=`                |
| TOOLS_CALLED      | `tools_called=`           |
| EXPECTED_TOOLS    | `expected_tools=`         |

If a parameter is listed in `evaluation_params` but missing from the test case, DeepEval will raise an error.

---

### Typical Pattern

```python
test_case = LLMTestCase(
    input=query,
    actual_output=response,
    expected_output=ground_truth,
    retrieval_context=retrieved_chunks
)

metrics = [
    AnswerRelevancyMetric(),
    FaithfulnessMetric(),
    GEval(
        name="Business Rule Check",
        criteria="Verify that the response contains a clear recommendation.",
        evaluation_params=[
            LLMTestCaseParams.ACTUAL_OUTPUT
        ]
    )
]

evaluate([test_case], metrics)
```

* Built-in metrics use the fields they require.
* G-Eval uses only the fields listed in its `evaluation_params`.
* All metrics can share the **same `LLMTestCase`**; you do **not** need to create separate test cases for each metric.
